# 02 — Feature Exploration

Tujuan: melihat apa yang benar-benar dimakan ranker, terutama Clean V2 cross-sectional + market context features.

Gunakan historical feature table, bukan fresh-forward outcome data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FEATURE_TABLE = Path(r"CHANGE_ME")

In [ ]:
if not FEATURE_TABLE.exists():
    raise FileNotFoundError("Set FEATURE_TABLE ke historical feature parquet/CSV lokal.")

df = pd.read_parquet(FEATURE_TABLE) if FEATURE_TABLE.suffix.lower() == ".parquet" else pd.read_csv(FEATURE_TABLE)
print(df.shape)

## Clean V2 feature family

In [ ]:
V2_XS_SOURCE = [
    "close_return_5", "close_return_20", "atr14_over_close",
    "close_position_20", "distance_high_20_atr", "distance_low_20_atr",
    "distance_high_60_atr", "distance_low_60_atr",
    "relative_volume_20", "log_regular_value_relative_20",
]
V2_XS = [f"xs_rank_{c}" for c in V2_XS_SOURCE]
V2_MARKET_CONTEXT = [
    "market_primary_liquid_count",
    "market_breadth_return_5_positive",
    "market_breadth_return_20_positive",
    "market_median_close_return_5",
    "market_median_close_return_20",
    "market_median_atr14_over_close",
    "market_median_close_position_20",
    "market_median_relative_volume_20",
    "market_median_log_regular_value_relative_20",
]
V2_MARKET_RELATIVE = [
    "market_relative_close_return_5",
    "market_relative_close_return_20",
    "market_relative_atr14_over_close",
    "market_relative_close_position_20",
    "market_relative_relative_volume_20",
    "market_relative_log_regular_value_relative_20",
]
V2_FEATURES = V2_XS + V2_MARKET_CONTEXT + V2_MARKET_RELATIVE

present = [c for c in V2_FEATURES if c in df.columns]
missing = [c for c in V2_FEATURES if c not in df.columns]
print("present:", len(present), "/ 25")
print("missing:", missing)

In [ ]:
if present:
    display(df[present].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T)

## Cross-section satu hari

In [ ]:
if "date" not in df or "ticker" not in df:
    raise KeyError("Feature table perlu ticker dan date.")

dates = pd.to_datetime(df["date"], errors="coerce")
DAY = dates.dropna().max()
day = df.loc[dates.eq(DAY)].copy()
print("day:", DAY, "rows:", len(day))

cols = ["ticker"] + [c for c in [
    "xs_rank_close_return_20",
    "xs_rank_atr14_over_close",
    "market_relative_close_return_20",
    "market_breadth_return_20_positive",
] if c in day.columns]
display(day[cols].sort_values(cols[1], ascending=False).head(20) if len(cols) > 1 else day.head())

## Distribution check

In [ ]:
for col in [c for c in ("xs_rank_close_return_20", "xs_rank_atr14_over_close", "market_relative_close_return_20") if c in df]:
    ax = df[col].dropna().hist(bins=50, figsize=(8, 3))
    ax.set_title(col)
    plt.show()

## Yang perlu dipahami

- `xs_rank_*` membandingkan saham dengan saham lain pada **tanggal yang sama**.
- `market_median_*` / breadth adalah kondisi cross-section market pada hari itu.
- `market_relative_*` adalah nilai saham dikurangi konteks market.
- Feature table bukan prediction; ini representation yang baru masuk ke estimator.